# Week 1 Day 2 Lab - Introducing UnderCover

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: larger models and Jev

PART 4: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls paid models through OpenRouter and costs only a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week1day2free.ipynb">week1day2free.ipynb</a>.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it! There's some interesting content on Jev for the pros.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a viking."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

response = openrouter.chat.completions.create(model="openai/gpt-6-luna", messages=messages)
display(Markdown(response.choices[0].message.content))

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

By default we work with the queue of 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

In [ ]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

In [ ]:
claims[0]

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [ ]:
labels[claims[0]["claim_id"]]

### Explore the data

It's always good to dig into the data before running experiments.

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

Claim CLM-1015 might be a familiar example...

In [ ]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

Here's another one that might ring a bell...

In [ ]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

This chart counts the claims of each type, split by the handler's decision.

In [ ]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [ ]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with some LLMs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [ ]:
def decide(claim, model, effort):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide(claims[0], "openai/gpt-6-luna", "low")
first

### Now running multiple models

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [ ]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

In [ ]:
luna_medium = run("openai/gpt-6-luna", "medium", claims, labels)

In [ ]:
grok = run("x-ai/grok-4.6", "low", claims, labels)

In [ ]:
luna_high = run("openai/gpt-6-luna", "high", claims, labels)

In [ ]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

## Results by difficulty

This counts the claims each entry got right at each level of difficulty. Expect nearly all of the manual claims to be wrong for now.

In [ ]:
scored = pd.DataFrame(luna_low + luna_medium + luna_high + gemini + grok)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Here we look at Luna at `low`. Pass in another set of results to see its misses.

In [ ]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(luna_low, labels)

### What it cost

This shows how many claims each entry got right, what it cost and how long it took for the whole queue.

In [ ]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Optional extras

These two experiments on the queue are optional. Nothing later in the lab depends on them.

### Sol at high

GPT-6 Sol is Luna's bigger sibling; we run it at the `high` reasoning level to see how much better it does on this prompt.

In [ ]:
sol = run("openai/gpt-6-sol", "high", claims, labels)

### Jev

Jev, from TypeSafe AI, doesn't generate text. We send it the facts as `state` along with a typed question, and it returns a probability for each option and a confidence. It chooses between approve, decline and refer in a fraction of a second. It can't write a rationale or work out a payout, so we score it on the decision alone.

Jev is on OpenRouter's Decisions endpoint, which the OpenAI client can't reach. We use OpenRouter's own Python client with the same key.

In [ ]:
JEV = "typesafe/jev-1.13"
JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    "criteria": {
        "approve": "The wording covers the claim and the required documents are attached.",
        "decline": "The wording does not cover the claim.",
        "refer": "A colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.",
    },
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def decide_with_jev(claim):
    """Ask Jev for the decision on one claim, with its confidence."""
    state = {"policy_wording": policy_wording, "policy": policies[claim["policy_id"]], "claim": claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
    answer = response.answers["decision"]
    return {
        "claim_id": claim["claim_id"],
        "decision": answer.choice,
        "confidence": answer.confidence,
        "right": answer.choice == labels[claim["claim_id"]]["decision"],
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 2),
    }

In [ ]:
jev = pd.DataFrame([decide_with_jev(claim) for claim in claims])
luna_decisions_right = sum(result["decision"] == labels[result["claim_id"]]["decision"] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for ${jev['cost'].sum():.4f} in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Here are Jev's decisions split at a confidence of 0.9. One way to use a fast model like this is to keep the answers it's confident about and send the rest to a stronger model.

In [ ]:
jev["confident"] = jev["confidence"] >= 0.9
jev.groupby("confident")["right"].agg(["sum", "count"])

## Part 4: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

In [ ]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses Luna at `low` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with Luna at `low`.

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)
nemotron = ChatOpenAI(model="nvidia/nemotron-3.5-lightning", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)
minimax = ChatOpenAI(model="minimax/minimax-m3.1-flash:free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
agent_nemotron = create_agent(model=nemotron, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
agent_minimax = create_agent(model=minimax, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)


First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

In [ ]:
result = agent_nemotron.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

In [ ]:
result = agent_minimax.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

Now the whole queue. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. OpenRouter's cost for each call is in the message metadata. A claim with no recorded decision counts as wrong.

In [ ]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
def decide_with_agent_nemotron(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent_nemotron.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "nvidia/nemotron-3.5-lightning",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_nemotron_results = []
for claim in claims:
    agent_nemotron_results.append(decide_with_agent_nemotron(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
def decide_with_agent_minimax(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent_nemotron.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "minimax/minimax-m3.1-flash",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_minimax_results = []
for claim in claims:
    agent_minimax_results.append(decide_with_agent_minimax(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
pd.DataFrame(luna_low + agent_results + agent_nemotron_results + agent_minimax_results + grok).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

In [ ]:
show_misses(agent_results, labels)

## Putting it all together

Here is every experiment in one table. Cost is for the whole queue and speed is the average time for one claim. Jev isn't included because it only makes the decision.

In [ ]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low, luna_medium, luna_high, grok, gemini, sol, agent_results, agent_nemotron_results, agent_minimax_results]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand the accuracy, cost and speed.<br/><br/>
            Based on our initial experiments, Luna (low) is able to perform well at a low cost. More expensive models can perform better.<br/><br/>
            Challenge: with a goal of at least 15 / 25 correct consistently, using the agent, what is the cheapest model you can find?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

OpenRouter Benchmark: The 20 Cheapest Tool-Calling Models
Objective
Challenge: Using the existing insurance-claims agent, find the least expensive model that correctly resolves at least 15 out of 25 claims consistently.

This is the English-language version of the original six-code-cell benchmark notebook. It keeps the original workflow, agent configuration, evaluation rules, and comparison with earlier experiments. The changes are:
- Spanish identifiers, output-column names, status labels, code comments, and printed messages have been translated into English.
- Explanatory Markdown has been added around the same six executable stages, so you can understand what each stage does and why it is needed.
- Original English interfaces (get_policy, record_decision, claims, labels, recorded, is_right, and the output fields model, right, cost, seconds) have been preserved.
Important: This notebook is an extension of your existing challenge notebook, not a replacement for its problem data and tools. Run your original setup cells first. The following objects must already exist:
Required object	What it is used for
openrouter_api_key	OpenRouter API authentication.
INSTRUCTIONS and policy_wording	The business instructions and insurance policy rules.
get_policy	Tool that retrieves the customer's policy.
record_decision	Tool that stores the agent's decision and payout.
claims	The claims to be tested. The benchmark uses the first 25.
labels	Ground-truth decisions indexed by claim_id.
recorded	Dictionary containing decisions saved by the tool.
is_right(answer, label)	Your existing grading function.


Run order: discover models → construct agents → inspect one claim → define scoring → run all 25 claims → repeat qualifying models → compare results.

This notebook calls live models and can incur API charges. A single claim can involve multiple LLM requests because the agent may need to call both tools and then continue reasoning.


1. Discover and rank eligible OpenRouter models
Celda 2 de 14

1. Discover and rank eligible OpenRouter models
What this cell does
1. Queries OpenRouter's /api/v1/models catalog with your API key.
2. Filters out models that do not advertise the tools parameter, do not produce text, have an insufficient context window, or are not suitable for a normal synchronous request. It also excludes special openrouter/ routing entries and :batch variants.
3. Reads the listed input, output, and per-request prices. These prices are supplied by the catalog, rather than hard-coded into the notebook.
4. Sorts eligible models by the simple average of input and output prices per million tokens, then selects the first 20.
5. Creates model_ids_20 (a plain list of model IDs) and model_info (lookup information used by later cells).
Settings you can change
Setting	Default	Meaning
PAID_ONLY	False	Include free models; set to True to exclude zero-priced entries and :free variants.
MIN_CONTEXT_TOKENS	0	Minimum advertised context length; e.g. 8192 to impose a minimum.
MODEL_COUNT	20	Number of candidate models.


Price interpretation: OpenRouter reports per-token charges. Multiplying a token price by 1,000,000 is only for human-readable comparison. The average of the two rates is a catalog ranking heuristic, not a prediction of total agent cost: a model may use much more output than input, make more tool calls, or fail to finish.

Availability interpretation: Reporting tool support in the catalog does not guarantee that an endpoint is available for every account, region, or request. Later cells capture API errors such as 404 and 429.

In [ ]:
# Stage 1: retrieve OpenRouter's catalog and select the 20 cheapest eligible models.
import json
import time
import requests
import pandas as pd
from IPython.display import display
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
PAID_ONLY = True          # True: exclude free models.
MIN_CONTEXT_TOKENS = 0     # Optional: try 8192, 16000, etc.
MODEL_COUNT = 20

response = requests.get(
    f'{OPENROUTER_BASE_URL}/models',
    headers={'Authorization': f'Bearer {openrouter_api_key}'},
    params={'supported_parameters': 'tools', 'output_modalities': 'text'},
    timeout=30,
)
response.raise_for_status()

catalog = []
for model_data in response.json()['data']:
    model_id = model_data['id']
    supported_parameters = set(model_data.get('supported_parameters') or [])
    architecture = model_data.get('architecture') or {}
    pricing = model_data.get('pricing') or {}

    # Only keep models usable for this text-based, tool-calling agent.
    if (
        'tools' not in supported_parameters
        or not model_data.get('top_provider')
        or model_id.startswith('openrouter/')
        or model_id.endswith(':batch')
        or (PAID_ONLY and model_id.endswith(':free'))
        or int(model_data.get('context_length') or 0) < MIN_CONTEXT_TOKENS
        or 'text' not in architecture.get('input_modalities', ['text'])
        or 'text' not in architecture.get('output_modalities', ['text'])
    ):
        continue

    # Prices in the API response are denominated in USD per token/request.
    try:
        input_price = float(pricing['prompt'])
        output_price = float(pricing['completion'])
        request_price = float(pricing.get('request') or 0)
    except (KeyError, TypeError, ValueError):
        continue

    if PAID_ONLY and input_price == 0 and output_price == 0 and request_price == 0:
        continue
    if input_price < 0 or output_price < 0 or request_price < 0:
        continue

    available_efforts = (model_data.get('reasoning') or {}).get('supported_efforts') or []
    supports_low_reasoning = (
        'low' in available_efforts
        and ('reasoning_effort' in supported_parameters or 'reasoning' in supported_parameters)
    )

    catalog.append({
        'model': model_id,
        'name': model_data.get('name', model_id),
        'context_length': int(model_data.get('context_length') or 0),
        'input_usd_per_million': input_price * 1_000_000,
        'output_usd_per_million': output_price * 1_000_000,
        'pricing': {
            'prompt': input_price,
            'completion': output_price,
            'request': request_price,
        },
        'supports_low_reasoning': supports_low_reasoning,
        'supports_reasoning_effort': 'reasoning_effort' in supported_parameters,
    })

# Ranking is based on the average input/output list price, not observed agent cost.
catalog.sort(key=lambda item: (
    (item['input_usd_per_million'] + item['output_usd_per_million']) / 2,
    item['input_usd_per_million'],
    item['model'],
))

selected_models = catalog[:MODEL_COUNT]
model_ids_20 = [item['model'] for item in selected_models]
model_info = {item['model']: item for item in selected_models}

assert len(model_ids_20) == MODEL_COUNT, 'Fewer than 20 eligible models were found'
print('Selected OpenRouter models:')
print(model_ids_20)
display(pd.DataFrame(selected_models)[
    ['model', 'input_usd_per_million', 'output_usd_per_million', 'context_length']
])

2. Build a LangChain agent for each model and inspect one example
What this cell does
The original experiment built separate ChatOpenAI instances for Luna, Nemotron, and MiniMax. Here, build_llm() creates a configured ChatOpenAI instance for any selected OpenRouter model, and a dictionary comprehension builds one LangChain agent per model.

Every agent receives exactly the same:
- System prompt: INSTRUCTIONS, the instruction to use the tools, and policy_wording.
- Tools: get_policy and record_decision.
- Claim format: make_claim_prompt(claim) sends the claim as formatted JSON.
- Agent recursion limit: 14, which prevents an indefinite chain of model/tool steps.
Reasoning controls and token-usage metadata
reasoning_effort='low' is sent only if the catalog declares low reasoning support. When it does not, the model runs with its default reasoning behavior. This avoids sending an unsupported argument to every provider.

extra_body={'usage': {'include': True}} asks OpenRouter to include usage details where possible; the evaluator will also work with estimates based on token counts.
The small, real test at the end
The final lines execute the first claim with the first model and print the agent's message trace. Inspect the trace to see whether the model calls get_policy, invokes record_decision, and finishes correctly. Executing this cell makes a real API call and may modify recorded. The main evaluator clears the prior entry for each claim before testing it.

In [ ]:
# Stage 2: build models and agents; execute one example to inspect the tool calls.
def build_llm(model_details):
    """Create ChatOpenAI for one OpenRouter model using its advertised settings."""
    parameters = dict(
        model=model_details['model'],
        base_url=OPENROUTER_BASE_URL,
        api_key=openrouter_api_key,
        timeout=120,
        max_retries=1,
        extra_body={'usage': {'include': True}},
    )

    # Not every OpenRouter model accepts the same reasoning parameters.
    if model_details['supports_low_reasoning']:
        if model_details['supports_reasoning_effort']:
            parameters['reasoning_effort'] = 'low'
        else:
            parameters['extra_body']['reasoning'] = {'effort': 'low'}
    return ChatOpenAI(**parameters)


AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """Present just one claim to the agent as human-readable JSON."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agents_20 = {
    model_id: create_agent(
        model=build_llm(model_info[model_id]),
        tools=[get_policy, record_decision],
        system_prompt=AGENT_SYSTEM_PROMPT,
    )
    for model_id in model_ids_20
}

# Smoke test: use the first eligible model on the first claim and print its trace.
test_model = model_ids_20[0]
result = agents_20[test_model].invoke(
    {'messages': [{'role': 'user', 'content': make_claim_prompt(claims[0])}]},
    config={'recursion_limit': 14},
)
print('Smoke-test model:', test_model)
for message in result['messages'][1:]:
    message.pretty_print()

3. Evaluate a single claim and calculate agent cost
Why a dedicated evaluator is necessary
The agent does more than generate text: it looks up a policy, decides whether a claim should be approved, and should save the final outcome through record_decision. We must grade the recorded decision, not merely whatever text the LLM returns.

decide_with_agent(claim, model_id, round_number=1) therefore:
1. Starts a timer and retrieves the expected answer from labels[claim_id].
2. Removes any previous entry for that claim from the recorded dictionary to avoid accidentally grading the decision from the last model. (This clears the in-memory dictionary; it does not undo any other persistence inside your tools.)
3. Executes the chosen model's agent and gives it access to the tools.
4. Reads the saved decision from recorded. If the agent never records one, it substitutes nothing recorded.
5. Calls your original is_right(answer, label) grading function and records its Boolean result in right.
6. Calculates elapsed time and the combined cost of all AI messages produced during the claim.
7. Catches request/tool exceptions and returns an error row rather than losing the rest of the benchmark.
How calculate_cost() works
Some providers return a reported cost in the response metadata (token_usage.cost or cost). When that is not available, the function estimates cost from prompt_tokens, completion_tokens, and the model's list prices. If any AI call is missing both a reported cost and token counts, the total becomes None and its source is unknown.

The cost_source column can be actual (all calls had reported costs), estimated (at least one estimate was required), or unknown. An estimated price is not the same as a final billed amount.

The returned row keeps the same model, effort, technique, right, cost, and seconds columns as your earlier notebook, so the results remain easy to concatenate.

In [ ]:
# Stage 3: calculate total LLM cost and evaluate one claim for any selected model.
def calculate_cost(result, pricing):
    """Return (total_usd, source) for every AI message in one agent execution."""
    total_cost = 0.0
    cost_sources = []

    for message in result['messages']:
        if message.type != 'ai':
            continue

        metadata = message.response_metadata or {}
        token_usage = metadata.get('token_usage') or {}
        reported_cost = token_usage.get('cost')
        if reported_cost is None:
            reported_cost = metadata.get('cost')

        if reported_cost is not None:
            total_cost += float(reported_cost)
            cost_sources.append('actual')
            continue

        usage = message.usage_metadata or {}
        input_tokens = token_usage.get('prompt_tokens', usage.get('input_tokens'))
        output_tokens = token_usage.get('completion_tokens', usage.get('output_tokens'))
        if input_tokens is None or output_tokens is None:
            cost_sources.append('unknown')
            continue

        total_cost += (
            input_tokens * pricing['prompt']
            + output_tokens * pricing['completion']
            + pricing['request']
        )
        cost_sources.append('estimated')

    if not cost_sources or 'unknown' in cost_sources:
        return None, 'unknown'
    return total_cost, ('actual' if all(source == 'actual' for source in cost_sources) else 'estimated')


def decide_with_agent(claim, model_id, round_number=1):
    """Run one agent on one claim; grade the saved decision and return its metrics."""
    started_at = time.time()
    claim_id = claim['claim_id']
    recorded.pop(claim_id, None)  # Do not reuse a prior model's stored answer.
    details = model_info[model_id]
    label = labels[claim_id]

    try:
        result = agents_20[model_id].invoke(
            {'messages': [{'role': 'user', 'content': make_claim_prompt(claim)}]},
            config={'recursion_limit': 14},
        )
        answer = recorded.get(claim_id, {
            'decision': 'nothing recorded', 'payout': 0.0, 'rationale': ''
        }).copy()
        cost, cost_source = calculate_cost(result, details['pricing'])
        status = 'ok' if claim_id in recorded else 'no_decision'

        return {
            'model': model_id,
            'effort': 'low' if details['supports_low_reasoning'] else 'default',
            'technique': 'Agent',
            'round': round_number,
            'claim_id': claim_id,
            **answer,
            'difficulty': label['difficulty'],
            'right': bool(is_right(answer, label)),
            'cost': cost,
            'cost_source': cost_source,
            'seconds': round(time.time() - started_at, 1),
            'status': status,
            'error': '',
        }
    except Exception as error:
        return {
            'model': model_id,
            'effort': 'low' if details['supports_low_reasoning'] else 'default',
            'technique': 'Agent',
            'round': round_number,
            'claim_id': claim_id,
            'difficulty': label['difficulty'],
            'decision': 'error',
            'payout': 0.0,
            'rationale': '',
            'right': False,
            'cost': None,
            'cost_source': 'unknown',
            'seconds': round(time.time() - started_at, 1),
            'status': 'api_error',
            'error': str(error)[:500],
        }

4. Run the first 25-claim benchmark for all 20 models
What this cell does
- Sets CLAIMS_PER_MODEL = 25.
- Runs the same first 25 claims through each agent in model_ids_20, one model at a time.
- Stores one result dictionary per attempt in benchmark_results.
- Stops testing the current model early if a serious API error is detected (401, 402, 403, 404, or 429), then moves to the next model.
- Summarizes each model's first round into pilot_report.
The first full pass can involve up to 20 × 25 = 500 agent executions, potentially with several LLM calls per execution. Each attempt writes through your existing record_decision tool if the agent chooses to call it.
How to read pilot_report
Column	Meaning
correct	Number of claims graded right=True. The challenge threshold is 15.
attempts	Number of attempted claims; fewer than 25 means testing stopped or insufficient input.
api_errors	Number of attempts that raised an exception.
no_decisions	Completed agent executions that never saved a decision.
cost_usd	Total measurable cost of the attempts (reported or estimated).
seconds	Total wall-clock execution time in seconds.


Caution: A model may have cost_usd missing or incomplete when its provider does not supply enough usage data. Do not interpret a missing cost as zero. The first-round ranking sorts by accuracy first, not price.

In [ ]:
# Stage 4: run the initial 25-claim benchmark and summarize each model.
CLAIMS_PER_MODEL = 25
benchmark_results = []

for model_id in model_ids_20:
    print('Testing:', model_id)
    for claim in claims[:CLAIMS_PER_MODEL]:
        row = decide_with_agent(claim, model_id, round_number=1)
        benchmark_results.append(row)

        # Do not burn through the remaining claims after a blocking API error.
        if row['status'] == 'api_error' and any(
            code in row['error'].lower() for code in ['404', '429', '401', '402', '403']
        ):
            print('Stopped this model:', row['error'][:180])
            break


def summarize_by_round(rows):
    """Aggregate detailed claim results into model-by-round totals."""
    frame = pd.DataFrame(rows)
    return (frame.groupby(['model', 'round'], as_index=False)
        .agg(
            correct=('right', 'sum'),
            attempts=('claim_id', 'count'),
            api_errors=('status', lambda values: (values == 'api_error').sum()),
            no_decisions=('status', lambda values: (values == 'no_decision').sum()),
            cost_usd=('cost', lambda values: values.sum(min_count=1)),
            seconds=('seconds', 'sum'),
        )
    )


pilot_report = summarize_by_round(benchmark_results)
print('Pilot results (highest accuracy first):')
print(pilot_report.sort_values(['correct', 'cost_usd'], ascending=[False, True]).to_string(index=False))

5. Repeat qualifying models to test consistency
Why the challenge needs multiple runs
A model passing 15/25 once may have benefited from randomness. This stage tests the same 25 claims two more times for models that meet the threshold in the pilot round.

The selection rule for finalists is unchanged from the original notebook:
- Exactly 25 claims were attempted in round 1.
- At least 15 of those claims were correct.
- No API errors occurred in round 1.
Each finalist is then run in rounds 2 and 3. The combined table round_report holds one row per model and round; summary holds one row per model across its attempted rounds.

A model has passed=True only when it completed three rounds, attempted 25 claims in each, obtained at least 15 correct in its worst round, and had no API errors. average_cost_per_25_usd is total observed cost divided by the number of rounds attempted.

Finally, the notebook saves the detailed results to benchmark_openrouter_20.csv, with one CSV row per claim attempt, so you can inspect decisions, failure reasons, cost sources, and elapsed time.

Statistical limitation: Repeating the same 25 claims checks run-to-run consistency on this exercise. It does not prove that the model generalizes to new claims or that its performance is stable across future model/provider updates.

In [ ]:
# Stage 5: repeat successful pilot models and identify consistent performers.
finalists = pilot_report.query(
    'attempts == @CLAIMS_PER_MODEL and correct >= 15 and api_errors == 0'
)['model'].tolist()

for round_number in [2, 3]:
    for model_id in finalists:
        print('Confirming round', round_number, 'for', model_id)
        for claim in claims[:CLAIMS_PER_MODEL]:
            row = decide_with_agent(claim, model_id, round_number=round_number)
            benchmark_results.append(row)

            if row['status'] == 'api_error' and any(
                code in row['error'].lower() for code in ['404', '429', '401', '402', '403']
            ):
                print('Stopped this model:', row['error'][:180])
                break

round_report = summarize_by_round(benchmark_results)
summary = (round_report.groupby('model', as_index=False)
    .agg(
        rounds=('round', 'nunique'),
        min_correct=('correct', 'min'),
        mean_correct=('correct', 'mean'),
        total_cost_usd=('cost_usd', lambda values: values.sum(min_count=1)),
        total_seconds=('seconds', 'sum'),
        api_errors=('api_errors', 'sum'),
        min_attempts=('attempts', 'min'),
    )
)

summary['passed'] = (
    (summary['rounds'] == 3)
    & (summary['min_attempts'] == CLAIMS_PER_MODEL)
    & (summary['min_correct'] >= 15)
    & (summary['api_errors'] == 0)
)
summary['average_cost_per_25_usd'] = summary['total_cost_usd'] / summary['rounds']

print('Models consistently scoring at least 15/25:')
display(summary[summary['passed']].sort_values('average_cost_per_25_usd'))

# Save claim-level results (including recorded decisions and failures) for analysis.
pd.DataFrame(benchmark_results).to_csv('benchmark_openrouter_20.csv', index=False)

6. (Optional) Compare against your earlier experiments
Your original challenge notebook already had result lists called luna_low, agent_results, agent_nemotron_results, agent_minimax_results, and grok.

This is the same optional comparison code from the original six-cell notebook, with the new English variable names. It is intentionally commented out because these earlier variables may not exist in every session.

Why use only round 1? Your old result lists were produced as separate experiments. Concatenating all three new rounds would give the 20-model benchmark more attempts and make the summed right, cost, and seconds columns incomparable. Filtering for round == 1 keeps the aggregation closer to your original comparison.

How to use it: Remove the leading # from each line once your earlier result lists are available in the notebook.

In [ ]:
# Stage 6 (optional): append the first benchmark round to earlier experiments.
first_round_results = [row for row in benchmark_results if row['round'] == 1]
combined_results = (
    luna_low
    + agent_results
    + grok
    + first_round_results
)
display(
    pd.DataFrame(combined_results)
    .groupby(['model', 'effort', 'technique'], sort=False)[['right', 'cost', 'seconds']]
    .sum()
)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You compared several models and an agent on UnderCover's claims, with accuracy, cost and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>